In [25]:
import pandas as pd

df = pd.read_excel("WHO.xlsx")

file_path = "JMP.xlsx"

wat = pd.read_excel(file_path, sheet_name="wat")
san = pd.read_excel(file_path, sheet_name="san")
hyg = pd.read_excel(file_path, sheet_name="hyg")
mh  = pd.read_excel(file_path, sheet_name="mh")

drop_cols = [
    'blank', 'name2', 'year2',
    'sl', 'latestyear', 'iso3',
    'region_sdg', 'region_unicef_programme', 'region_who',
    'region_lldc', 'region_ldc', 'region_sids', 'region_fragileoecdany',
    'region_income',
    'name_AR', 'name_ES', 'name_FR', 'name_RU', 'name_ZH'
]

for sheet in [wat, san, hyg, mh]:
    sheet.drop(columns=[c for c in drop_cols if c in sheet.columns], inplace=True)


In [26]:
target_countries = df['Country'].unique()

country_mapping = {
    "Central African Republic": "Central African Rep",
    "Côte d'Ivoire": "Cote d'Ivoire",
    "Czechia": "Czech Republic",
    "Democratic Republic of the Congo": "Dem Rep Congo",
    "Iran (Islamic Republic of)": "Iran",
    "Lao People's Democratic Republic": "Laos",
    "Micronesia (Federated States of)": "Micronesia",
    "Republic of Korea": "Korea (Rep of)",
    "United Republic of Tanzania": "Tanzania",
    "United States of America": "USA",
    "Venezuela (Bolivarian Republic of)": "Venezuela",
    "Viet Nam": "Vietnam"
}

for i in [wat, san, hyg, mh]:
    i['name'] = i['name'].replace(country_mapping)
    i.drop(i[~i['name'].isin(target_countries)].index, inplace=True)

In [27]:
years = list(range(2000, 2025))

skeleton = pd.MultiIndex.from_product(
    [target_countries, years],
    names=['name', 'year']
).to_frame(index=False)

merged_df = skeleton.copy()

merged_df = merged_df.merge(wat, on=['name', 'year'], how='left')
merged_df = merged_df.merge(san, on=['name', 'year'], how='left', suffixes=('', '_san'))
merged_df = merged_df.merge(hyg, on=['name', 'year'], how='left', suffixes=('', '_hyg'))
merged_df = merged_df.merge(mh,  on=['name', 'year'], how='left', suffixes=('', '_mh'))

for base_col in ['pop_t', 'prop_u']:
    dupe_cols = [c for c in merged_df.columns if c == base_col or c.startswith(base_col + '_san')
                 or c.startswith(base_col + '_hyg') or c.startswith(base_col + '_mh')]
    if len(dupe_cols) > 1:
        coalesced = merged_df[dupe_cols[0]]
        for c in dupe_cols[1:]:
            coalesced = coalesced.combine_first(merged_df[c])
        merged_df[base_col] = coalesced
        merged_df.drop(columns=[c for c in dupe_cols if c != base_col], inplace=True)

In [28]:
merged_df.rename(columns={'name': 'Country', 'year': 'Year'}, inplace=True)
df = df.drop(columns=['Total_Cases', 'Imported_Cases', 'Deaths'])

final_df = df.merge(merged_df, on=['Country', 'Year'], how='right')

if 'WHO_Region' in final_df.columns:
    final_df['WHO_Region'] = final_df.groupby('Country')['WHO_Region'].transform(lambda x: x.ffill().bfill())

final_df.to_csv("MERGED.csv", index=False)

print(final_df.shape)
final_df.columns.tolist()

(3150, 129)


['Country',
 'Year',
 'WHO_Region',
 'Autochthonous_Cases',
 'pop_t',
 'prop_u',
 'wat_basal_r',
 'wat_lim_r',
 'wat_unimp_r',
 'wat_ns_r',
 'arc_wat_basal_r',
 'wat_basal_u',
 'wat_lim_u',
 'wat_unimp_u',
 'wat_ns_u',
 'arc_wat_basal_u',
 'wat_basal_t',
 'wat_lim_t',
 'wat_unimp_t',
 'wat_ns_t',
 'arc_wat_basal_t',
 'wat_sm_r',
 'wat_imp_prem_r',
 'wat_imp_av_r',
 'wat_imp_qual_r',
 'wat_arc_sm_r',
 'wat_pip_r',
 'wat_imp_npip_r',
 'wat_sm_u',
 'wat_imp_prem_u',
 'wat_imp_av_u',
 'wat_imp_qual_u',
 'wat_arc_sm_u',
 'wat_pip_u',
 'wat_imp_npip_u',
 'wat_sm_t',
 'wat_imp_prem_t',
 'wat_imp_av_t',
 'wat_imp_qual_t',
 'wat_arc_sm_t',
 'wat_pip_t',
 'wat_imp_npip_t',
 'san_basal_r',
 'san_lim_r',
 'san_unimp_r',
 'san_ns_r',
 'arc_san_basal_r',
 'arc_san_ns_r',
 'san_basal_u',
 'san_lim_u',
 'san_unimp_u',
 'san_ns_u',
 'arc_san_basal_u',
 'arc_san_ns_u',
 'san_basal_t',
 'san_lim_t',
 'san_unimp_t',
 'san_ns_t',
 'arc_san_basal_t',
 'arc_san_ns_t',
 'san_sm_r',
 'san_sdo_sm_r',
 'san_fst_